# Teaching Module: Financial Derivatives with Python
**Henry Mayambu | MSc Financial Engineering | For Finance Lecturer - Nov 2026**

In [ ]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

S0=3500; r=0.08; T=0.5
F=S0*np.exp(r*T)
print(f'6M Futures: {F:.2f} ZAR')

## Black-Scholes - S=450 Anglo American, K=470, r=8.25% SA

In [ ]:
def bs(S,K,T,r,sig,typ='call'):
    d1=(np.log(S/K)+(r+0.5*sig**2)*T)/(sig*np.sqrt(T))
    d2=d1-sig*np.sqrt(T)
    if typ=='call': return S*norm.cdf(d1)-K*np.exp(-r*T)*norm.cdf(d2)
    else: return K*np.exp(-r*T)*norm.cdf(-d2)-S*norm.cdf(-d1)

print('Call:', bs(450,470,0.25,0.0825,0.35,'call'))
print('Put:', bs(450,470,0.25,0.0825,0.35,'put'))

## Greeks - Delta, Gamma, Vega

In [ ]:
S_range=np.linspace(300,600,100)
deltas=[norm.cdf((np.log(s/470)+(0.0825+0.5*0.35**2)*0.25)/(0.35*np.sqrt(0.25))) for s in S_range]
plt.plot(S_range,deltas); plt.title('Delta Call vs Price'); plt.grid(); plt.show()

## Binomial American - Shows early exercise premium for Puts

In [ ]:
def binomial_american(S,K,T,r,sig,N,typ='call'):
    dt=T/N; u=np.exp(sig*np.sqrt(dt)); d=1/u; p=(np.exp(r*dt)-d)/(u-d)
    price=np.zeros((N+1,N+1))
    for i in range(N+1):
        for j in range(i+1): price[j,i]=S*(u**(i-j))*(d**j)
    opt=np.zeros((N+1,N+1))
    for j in range(N+1): opt[j,N]=max(0,price[j,N]-K) if typ=='call' else max(0,K-price[j,N])
    for i in range(N-1,-1,-1):
        for j in range(i+1):
            hold=np.exp(-r*dt)*(p*opt[j,i+1]+(1-p)*opt[j+1,i+1])
            ex=max(0,price[j,i]-K) if typ=='call' else max(0,K-price[j,i])
            opt[j,i]=max(hold,ex)
    return opt[0,0]

print('American Call:', binomial_american(450,470,0.25,0.0825,0.35,100,'call'))
print('American Put:', binomial_american(450,470,0.25,0.0825,0.35,100,'put'))